# Kaggle 训练阶段 2：恢复并完成第 15 轮

本 Notebook 使用阶段 1 的相同 run-id 和 AList 路径，要求恢复第 5 轮检查点，并继续训练到 15 epoch。恢复缺失或不兼容时立即失败。

In [ ]:
import os

os.environ.setdefault('DL_HELPER_GIT_REPO', 'https://github.com/lhiqwj173/dl_helper.git')
os.environ.setdefault('DL_HELPER_GIT_REF', '21e4c423f776f1b30186a52556d29d4d44086a0e')
os.environ.setdefault('DL_HELPER_MNIST_PATH', '/kaggle/input/datasets/vikramtiwari/mnist-numpy/mnist.npz')
os.environ.setdefault('DL_HELPER_RUN_ID', 'mnist-15epoch-stage1')
os.environ.setdefault('ALIST_HOST', 'http://139.196.47.52')
os.environ.setdefault('ALIST_BASE_PATH', '/dl-helper/mnist-15epoch')
os.environ.setdefault('WECOM_TO_USER', '@all')
print('恢复 run-id:', os.environ['DL_HELPER_RUN_ID'])

In [ ]:
import os, subprocess, sys

repo_dir = '/kaggle/working/dl-helper'
if os.path.exists(repo_dir):
    raise RuntimeError(f'目录已存在，请新建 Kaggle Session 后重试: {repo_dir}')

def checked(argv, *, cwd=None):
    proc = subprocess.run(argv, cwd=cwd, capture_output=True, text=True, encoding='utf-8')
    if proc.stdout:
        print(proc.stdout, end='')
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr, file=sys.stderr, end='')
        raise RuntimeError(f'命令失败，退出码 {proc.returncode}: {argv}')
    return proc

checked(['git', 'clone', os.environ['DL_HELPER_GIT_REPO'], repo_dir])
checked(['git', 'checkout', os.environ['DL_HELPER_GIT_REF']], cwd=repo_dir)
head = checked(['git', 'rev-parse', 'HEAD'], cwd=repo_dir).stdout.strip()
if head.lower() != os.environ['DL_HELPER_GIT_REF'].lower():
    raise RuntimeError(f'checkout HEAD 不匹配: {head}')
os.environ['DL_HELPER_REPO_DIR'] = repo_dir
checked([sys.executable, f'{repo_dir}/envs/kaggle_bootstrap.py'], cwd=repo_dir)
print('[bootstrap] fixed revision:', head)

In [ ]:
from pathlib import Path
import yaml

source = Path('/kaggle/working/dl-helper/examples/configs/kaggle/mnist.yaml')
config_path = Path('/kaggle/working/dl-helper-stage2.yaml')
with source.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
config['experiment']['data_path'] = os.environ['DL_HELPER_MNIST_PATH']
config['training']['max_epochs'] = 15
config['backend']['torch']['mixed_precision'] = 'no'
config['distributed']['num_processes'] = 1
config['selection']['patience'] = 30
config['checkpoint']['every_epochs'] = 1
config['checkpoint']['keep_last'] = 2
config['run']['id'] = os.environ['DL_HELPER_RUN_ID']
config['run']['source_revision'] = os.environ['DL_HELPER_GIT_REF']
config['remote'] = {'type': 'alist', 'host': os.environ['ALIST_HOST'], 'base_path': os.environ['ALIST_BASE_PATH'], 'user_secret_key': 'ALIST_USER', 'password_secret_key': 'ALIST_PWD', 'connect_timeout_seconds': 10, 'read_timeout_seconds': 60, 'max_attempts': 3, 'async_upload': False, 'failure_policy': 'required'}
config['notifications'] = {'type': 'wecom', 'corp_id_secret_key': 'WECOM_CORP_ID', 'corp_secret_key': 'WECOM_CORP_SECRET', 'agent_id_secret_key': 'WECOM_AGENT_ID', 'to_user': os.environ['WECOM_TO_USER'], 'connect_timeout_seconds': 10, 'read_timeout_seconds': 30, 'max_attempts': 3, 'failure_policy': 'required'}
with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)
print('配置已写入:', config_path)

In [ ]:
import json, subprocess, sys

preflight = subprocess.run([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(config_path), '--project-dir', '/kaggle/working/dl-helper/examples', '--experiment', 'experiments.mnist:build_experiment', '--resume', 'required', '--preflight-only'], cwd=repo_dir, text=True, encoding='utf-8')
if preflight.returncode != 0:
    raise RuntimeError(f'恢复预检失败，退出码 {preflight.returncode}')
run_proc = subprocess.run([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(config_path), '--project-dir', '/kaggle/working/dl-helper/examples', '--experiment', 'experiments.mnist:build_experiment', '--resume', 'required', '--run-id', os.environ['DL_HELPER_RUN_ID']], cwd=repo_dir, text=True, encoding='utf-8')
if run_proc.returncode != 0:
    raise RuntimeError(f'阶段 2 必须完成到第 15 轮，退出码: {run_proc.returncode}')
summary_path = Path('/kaggle/working/dl-helper-runs/runs') / os.environ['DL_HELPER_RUN_ID'] / 'metrics' / 'summary.json'
with summary_path.open('r', encoding='utf-8') as f:
    summary = json.load(f)
if summary['epoch'] != 15:
    raise RuntimeError(f"最终 epoch 必须为 15，实际为 {summary['epoch']}")
print('阶段 2 完成：训练已恢复并完成 15 epoch')